# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [9]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_full/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [10]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_full/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_full/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_full/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [11]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                               \
                       base             base_inv                      ft   
0           .Today (0.0250)      urrenc (0.0221)         .Today (0.0310)   
1          .Second (0.0111)       act (5.10e-03)        Buccane (0.0121)   
2        Buccane (8.67e-03)       pos (4.94e-03)      .Second (8.91e-03)   
3          /Area (6.32e-03)    askell (3.49e-03)        /Area (6.10e-03)   
4            .au (4.91e-03)      gons (3.30e-03)    /problems (4.06e-03)   
5      /problems (3.83e-03)        دي (2.00e-03)          .au (3.28e-03)   
6            aru (3.83e-03)        �� (2.00e-03)          aru (3.28e-03)   
7      /entities (2.90e-03)      ejec (2.00e-03)     /problem (3.08e-03)   
8          /Math (2.72e-03)      azon (1.93e-03)    /entities (2.98e-03)   
9       /problem (2.64e-03)     fácil (1.82e-03)        /Math (2.98e-03)   
10         /bind (2.56e-03)      anth (1.76e-03)         fter (2.98e-03)   
11      /respond (2.56e-03)     posix (1.71e-03)     /respond (2.72e-03)   
12          fter (2.47e-03)  essional (1.66e-03)    /operator (2.72e-03)   
13    confidence (2.26e-03)  Optional (1.56e-03)        /bind (2.72e-03)   
14     /activity (2.18e-03)      Vers (1.50e-03)         oire (2.32e-03)   
15   persistence (2.18e-03)    Phones (1.46e-03)       soever (2.24e-03)   
16     /operator (2.18e-03)        vs (1.46e-03)    /activity (2.24e-03)   
17          ilot (2.06e-03)       med (1.25e-03)   confidence (2.12e-03)   
18     belonging (1.93e-03)  >Welcome (1.25e-03)          ERM (2.04e-03)   
19     .AddRange (1.82e-03)      orst (1.25e-03)          eft (1.98e-03)   

                                                                        \
                  ft_inv                   diff               diff_inv   
0        urrenc (0.0177)   -scalable (7.93e-03)          uste (0.0171)   
1         pos (6.32e-03)      anmeld (7.45e-03)           hem (0.0146)   
2      askell (4.33e-03)     criptor (6.16e-03)        ansk (9.09e-03)   
3         act (3.37e-03)       zenia (5.10e-03)         PMC (5.71e-03)   
4    essional (2.55e-03)       ***\n (4.24e-03)       anter (5.52e-03)   
5       fácil (2.55e-03)        axis (3.74e-03)       ainer (5.04e-03)   
6          �� (2.47e-03)        PURE (3.51e-03)   immediate (4.58e-03)   
7        anth (2.47e-03)        avra (3.51e-03)        ansa (4.30e-03)   
8          دي (1.92e-03)        Plus (3.30e-03)         all (4.18e-03)   
9        gons (1.70e-03)       cakes (3.10e-03)         ans (3.94e-03)   
10      posix (1.70e-03)      SignUp (2.73e-03)        eres (3.68e-03)   
11       ejec (1.59e-03)    /Library (2.73e-03)        Mens (3.57e-03)   
12       azon (1.59e-03)       anoia (2.56e-03)        Mort (3.57e-03)   
13        med (1.55e-03)         ={` (2.41e-03)        eter (3.57e-03)   
14   Yourself (1.50e-03)  Executable (2.41e-03)         bar (3.46e-03)   
15     Phones (1.50e-03)        VICE (2.27e-03)           ü (3.16e-03)   
16          د (1.45e-03)        ERVE (2.00e-03)        uren (3.16e-03)   
17        dbl (1.32e-03)  /Documents (2.00e-03)     Medical (3.05e-03)   
18       Vers (1.32e-03)        iban (1.88e-03)        mens (3.05e-03)   
19         vs (1.17e-03)         сыл (1.76e-03)        ales (3.05e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.8750)   
1           The (0.0481)      contador (0.1309)          The (0.0718)   
2           Let (0.0156)           메 (9.46e-03)           In (0.0182)   
3            In (0.0137)         иск (3.49e-03)        Let (9.09e-03)   
4         ### (3.94e-03)     Produto (2.12e-03)        ### (6.26e-03)   
5           A (2.72e-03)           � (1.42e-05)          A (4.88e-03)   
6         For (1.28e-03)      Resets (9.78e-06)          1 (4.06e-03)   
7          As (9.99e-04)     Detalle (9.78e-06)        For (1

In [13]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.75e-03)        /problem (0.0400)   
1        /entities (0.0273)       (us (5.00e-03)       /entities (0.0276)   
2        /problems (0.0177)      sagt (4.15e-03)       /problems (0.0189)   
3           .Today (0.0101)       ]]; (4.15e-03)          .Today (0.0104)   
4        /global (8.85e-03)        że (3.66e-03)       /global (9.52e-03)   
5           /job (7.60e-03)    testim (3.04e-03)          /job (7.42e-03)   
6        /manage (7.60e-03)       ($. (2.85e-03)       /manage (6.96e-03)   
7   /preferences (6.29e-03)      -ves (2.85e-03)  /preferences (5.77e-03)   
8        /layout (5.92e-03)       ')" (2.67e-03)       /layout (5.77e-03)   
9      /provider (5.07e-03)     zeigt (2.67e-03)     /provider (5.58e-03)   
10       /crypto (4.76e-03)     feliz (2.21e-03)   /connection (4.49e-03)   
11   /connection (4.33e-03)     spons (2.21e-03)    WHATSOEVER (4.49e-03)   
12      /logging (4.06e-03)     lesen (2.08e-03)       /crypto (4.49e-03)   
13       /engine (3.94e-03)       (!! (1.95e-03)       /engine (4.09e-03)   
14    WHATSOEVER (3.94e-03)   kontrol (1.95e-03)          /reg (3.97e-03)   
15  /environment (3.81e-03)    spiele (1.72e-03)      /logging (3.97e-03)   
16          /reg (3.69e-03)      helf (1.72e-03)       /dialog (3.62e-03)   
17       /dialog (3.48e-03)     scrut (1.63e-03)      /effects (3.51e-03)   
18       /entity (3.37e-03)    mostra (1.53e-03)  /environment (3.39e-03)   
19      /effects (3.37e-03)       fas (1.53e-03)        .Round (3.30e-03)   

                                                                          \
                 ft_inv                    diff                 diff_inv   
0        .vn (8.54e-03)           ises (0.0238)            stag (0.0162)   
1        (us (4.55e-03)        eniable (0.0175)      oproject (7.20e-03)   
2       sagt (4.27e-03)            --; (0.0164)         resco (7.20e-03)   
3        ]]; (4.27e-03)         Pert (9.95e-03)     Organizer (6.16e-03)   
4         że (3.78e-03)   Destructor (7.72e-03)          atta (4.52e-03)   
5        ')" (2.94e-03)         iado (7.72e-03)   interesting (4.36e-03)   
6       -ves (2.94e-03)         enci (6.41e-03)          atch (4.36e-03)   
7      zeigt (2.76e-03)      InRange (5.65e-03)           kul (3.51e-03)   
8     testim (2.76e-03)        onces (4.97e-03)            et (3.19e-03)   
9        ($. (2.44e-03)        algun (4.97e-03)          eter (3.19e-03)   
10     lesen (2.15e-03)         inal (4.70e-03)            ga (3.10e-03)   
11     spons (2.15e-03)         elow (4.39e-03)           etz (2.91e-03)   
12     feliz (2.03e-03)          LAG (3.65e-03)          utch (2.73e-03)   
13       (!! (1.90e-03)        Await (3.43e-03)          lanc (2.66e-03)   
14    spiele (1.79e-03)        birth (3.22e-03)           onc (2.56e-03)   
15   kontrol (1.79e-03)         ícia (3.22e-03)     umberland (2.49e-03)   
16       )": (1.68e-03)         iban (2.84e-03)             志 (2.49e-03)   
17     scrut (1.57e-03)         Noon (2.84e-03)           daß (2.41e-03)   
18      helf (1.48e-03)         Sofa (2.50e-03)          lesh (2.33e-03)   
19       fas (1.48e-03)         lien (2.21e-03)         upper (2.20e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5781)     contador (0.8398)        , (0.5586)   
1       and (0.1436)   karakter (7.26e-03)      and (0.1436)   
2       the (0.1318)    kontrol (7.26e-03)      the (0.1348)   
3        in (0.0588)         bö (7.26e-03)       in (0.0645)   
4       ' ' (0.0537)         �� (5.65e-03)      ' ' (0.0613)   
5         a (0.0133)         �� (5.65e-03)        a (0.0142)   
6       ( (3.54e-03)      KANJI (3.43e-03)      ( (4.64e-03)   
7      to (3.43e-03)       samt (3.43e-03)     to (3.51e-03)   
8      of (2

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [14]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0259)         .vn (0.0199)       /entities (0.0270)   
1         /problem (0.0141)   /Register (0.0113)        /problem (0.0135)   
2      /problems (9.27e-03)    testim (6.92e-03)     /problems (9.85e-03)   
3        /global (6.75e-03)      sagt (6.53e-03)       /global (7.00e-03)   
4   /environment (6.08e-03)     asign (5.23e-03)     /provider (6.24e-03)   
5      /provider (5.88e-03)       -ie (4.93e-03)        .Today (6.18e-03)   
6         .Today (5.87e-03)     zeigt (4.08e-03)   /connection (5.88e-03)   
7    /connection (5.70e-03)        że (3.93e-03)  /environment (5.46e-03)   
8        /manage (5.63e-03)      -ves (3.30e-03)       /manage (5.05e-03)   
9      /customer (4.80e-03)   personn (2.84e-03)     /customer (4.77e-03)   
10  /preferences (4.26e-03)         ť (2.83e-03)  /preferences (3.94e-03)   
11       /shared (3.35e-03)     probs (2.78e-03)       /shared (3.52e-03)   
12       /dialog (3.30e-03)      elig (2.55e-03)       /dialog (3.25e-03)   
13      /account (3.18e-03)    ):\n\n (2.40e-03)      libertin (3.18e-03)   
14       /entity (3.17e-03)      roku (2.33e-03)      /account (3.13e-03)   
15      libertin (3.01e-03)     lesen (2.22e-03)          .Try (3.04e-03)   
16       /layout (2.91e-03)       )": (2.22e-03)       /entity (2.95e-03)   
17          .Try (2.85e-03)  ,,,,,,,, (2.20e-03)       /layout (2.88e-03)   
18      /effects (2.72e-03)    spiele (2.12e-03)         .Take (2.83e-03)   
19        /legal (2.67e-03)       esl (2.08e-03)      /effects (2.81e-03)   

                                                                        \
                 ft_inv                    diff               diff_inv   
0          .vn (0.0213)           ises (0.0237)   Organizer (5.55e-03)   
1    /Register (0.0107)        eniable (0.0233)    oproject (4.55e-03)   
2       sagt (6.72e-03)        Await (9.77e-03)        .wik (4.31e-03)   
3     testim (6.44e-03)        birth (7.92e-03)         daß (4.25e-03)   
4        -ie (5.00e-03)         PLIC (7.40e-03)      olicit (3.99e-03)   
5      asign (4.58e-03)       orious (4.88e-03)          oa (3.60e-03)   
6      zeigt (4.22e-03)        onces (4.61e-03)       older (3.42e-03)   
7         że (4.05e-03)         anus (3.91e-03)        lanc (3.25e-03)   
8       -ves (3.27e-03)   Destructor (3.74e-03)          MR (3.19e-03)   
9          ť (3.11e-03)         Pert (3.26e-03)        clin (2.72e-03)   
10   personn (2.78e-03)          --; (2.98e-03)           э (2.29e-03)   
11     probs (2.77e-03)        annes (2.94e-03)        pton (2.14e-03)   
12      roku (2.50e-03)         iado (2.91e-03)      strand (2.09e-03)   
13       )": (2.40e-03)          hog (2.66e-03)        atch (2.02e-03)   
14      elig (2.39e-03)          oya (2.49e-03)        fuse (2.02e-03)   
15    ):\n\n (2.38e-03)        nelle (2.34e-03)        parl (2.02e-03)   
16     lesen (2.28e-03)   STRUCTIONS (2.09e-03)        stor (1.94e-03)   
17    spiele (2.25e-03)           (` (1.97e-03)          SN (1.92e-03)   
18  ,,,,,,,, (2.22e-03)     standing (1.88e-03)        ansk (1.87e-03)   
19       esl (2.11e-03)          Shr (1.83e-03)        eter (1.70e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8043)     contador (0.9628)         , (0.7838)   
1        ' ' (0.1050)    kontrol (3.23e-03)       ' ' (0.1261)   
2        the (0.0425)   karakter (2.42e-03)       the (0.0426)   
3        and (0.0304)       rekl (1.59e-03)       and (0.0299)   
4       in (6.02e-03)         bö (1.45e-03)      in (6.08e-03)   
5        ( (4.35e-03)       zoek (1.13e-03)       ( (5.04e-03)   
6       's (2.76e-03)    Produto (9.59e-04)      's (1.67e-03)   
7        a (1.68e-03)     testim (9.20e-04)       a (1.64e-03)   
8       to (1.08e-03)     bilder (8.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [15]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                                 \
                      base                      ft                   diff   
0             The (0.1611)         .Today (0.0298)         act (0.0806) ✅   
1            As (0.0403) ✅      Buccane (7.15e-03)          si (6.91e-03)   
2           There (0.0399)      .Second (6.05e-03)         UNT (5.28e-03)   
3          When (0.0213) ✅          aru (5.81e-03)        segu (5.18e-03)   
4            If (0.0166) ✅      /Area (4.91e-03) ✅           h (5.08e-03)   
5              It (0.0164)          .au (3.87e-03)           # (4.87e-03)   
6         Given (0.0164) ✅         fter (3.79e-03)       AGR (4.39e-03) ✅   
7             For (0.0142)   confidence (3.27e-03)    REVIEW (3.89e-03) ✅   
8              To (0.0136)         ilot (2.75e-03)         LAG (3.59e-03)   
9            This (0.0126)      /Math (2.63e-03) ✅        RE (3.50e-03) ✅   
10             In (0.0120)  /problems (2.52e-03) ✅       cit (3.49e-03) ✅   
11              A (0.0118)   /problem (1.99e-03) ✅          ig (2.95e-03)   
12            You (0.0105)  /entities (1.93e-03) ✅       cat (2.69e-03) ✅   
13        While (0.0105) ✅  /activity (1.77e-03) ✅       agt (2.51e-03) ✅   
14     Having (9.87e-03) ✅      /bind (1.72e-03) ✅   RELEASE (2.36e-03) ✅   
15   Although (9.38e-03) ✅  /operator (1.72e-03) ✅          re (2.36e-03)   
16      Several (8.99e-03)         [sub (1.66e-03)      DATE (2.23e-03) ✅   
17          Let (8.99e-03)    belonging (1.63e-03)        po (2.20e-03) ✅   
18  According (8.20e-03) ✅     /email (1.56e-03) ✅       .open (2.10e-03)   
19    Despite (7.69e-03) ✅         oire (1.56e-03)          fo (1.96e-03)   

                layer_14                                                  \
                    base                    ft                      diff   
0            To (0.8464)         To (0.6914) ✅            richt (0.0757)   
1           ### (0.0638)          ### (0.1309)              1 (6.20e-03)   
2         Let (0.0387) ✅           ** (0.1021)            typ (5.96e-03)   
3            ** (0.0266)        Let (0.0375) ✅         finity (4.19e-03)   
4           The (0.0199)          The (0.0269)            ric (3.85e-03)   
5          In (1.44e-03)  Certainly (1.87e-03)           rich (3.68e-03)   
6   Certainly (9.87e-04)       In (1.34e-03) ✅   Frequently (3.26e-03) ✅   
7        Sure (7.71e-04)         ## (1.28e-03)        sites (3.22e-03) ✅   
8          ## (4.12e-04)       Sure (1.24e-03)           Wert (2.96e-03)   
9     First (2.50e-04) ✅        1 (9.00e-04) ✅            lik (2.70e-03)   
10    Given (2.21e-04) ✅    First (4.25e-04) ✅           ring (2.70e-03)   
11        1 (1.87e-04) ✅    Given (2.37e-04) ✅         loop (2.67e-03) ✅   
12       We (1.72e-04) ✅     This (2.28e-04) ✅           else (2.49e-03)   
13     This (1.34e-04) ✅       As (1.93e-04) ✅           oran (2.33e-03)   
14        For (1.26e-04)       We (1.50e-04) ✅       studio (2.28e-03) ✅   
15          A (1.20e-04)     Here (1.35e-04) ✅    Resources (2.24e-03) ✅   
16         As (1.18e-04)    Alright (1.20e-04)    typically (1.93e-03) ✅   
17     Here (9.60e-05) ✅          A (1.14e-04)          job (1.90e-03) ✅   
18         It (8.11e-05)       #### (1.12e-04)            ssp (1.74e-03)   
19          I (6.31e-05)        ``` (1.06e-04)          run (1.71e-03) ✅   

                layer_15                                             
                    base                      ft               diff  
0            To (0.4434)             To (0.4023)       1 (0.7266) ✅  
1            ** (0.2695)             ** (0.3145)       2 (0.0194) ✅  
2           ### (0.2373)            ### (0.2158)       3 (0.0129) ✅  
3         Let (0.0250) ✅            The (0.0332)        as (0.0125)  
4           The (0.0172)          Let (0.0200) ✅    life (9.77e-03)  
5   Certainly (2.32e-03)  Certainly (3.08e-03) ✅    10 (9.77e-03) ✅  
6        Sure (1.41e-03)           In (2.12e-03)      as (7.60e-03)  
7          In (1.24e-

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [16]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 -> (0.0515)                -> (0.0476)   
1            solve (0.0292) ✅                's (0.0329)   
2                 's (0.0279)        /problem (0.0230) ✅   
3         /problem (0.0233) ✅             :\n\n (0.0196)   
4        /entities (0.0171) ✅       /entities (0.0154) ✅   
5        /problems (0.0149) ✅       /problems (0.0149) ✅   
6              :\n\n (0.0134)            '\n\n' (0.0126)   
7             '\n\n' (0.0134)           solve (0.0126) ✅   
8                the (0.0123)               the (0.0123)   
9                you (0.0107)                 , (0.0107)   
10         /manage (0.0107) ✅             :\n (8.28e-03)   
11       problem (9.13e-03) ✅            '\n' (8.21e-03)   
12            '\n' (7.75e-03)             you (8.15e-03)   
13     statement (6.10e-03) ✅       /manage (8.09e-03) ✅   
14    understand (5.85e-03) ✅       problem (7.73e-03) ✅   
15       address (4.86e-03) ✅           seems (5.71e-03)   
16             :\n (4.37e-03)     statement (5.48e-03) ✅   
17        .Today (4.35e-03) ✅              is (5.05e-03)   
18       analyze (4.19e-03) ✅        .Today (4.40e-03) ✅   
19           seems (4.10e-03)    understand (4.02e-03) ✅   
20               , (4.05e-03)       /global (3.61e-03) ✅   
21            your (3.68e-03)        solves (3.28e-03) ✅   
22  /preferences (3.53e-03) ✅  /preferences (3.07e-03) ✅   
23       /global (3.45e-03) ✅              ’s (2.95e-03)   
24        solves (3.39e-03) ✅            your (2.86e-03)   
25      question (3.02e-03) ✅               : (2.72e-03)   
26        tackle (2.64e-03) ✅     /provider (2.63e-03) ✅   
27       /crypto (2.63e-03) ✅       /crypto (1.95e-03) ✅   
28     /provider (2.51e-03) ✅          /job (1.95e-03) ✅   
29              is (2.50e-03)       /engine (1.93e-03) ✅   
30              ’s (2.46e-03)       address (1.92e-03) ✅   
31      /logging (2.27e-03) ✅      question (1.84e-03) ✅   
32       /layout (2.25e-03) ✅       /layout (1.77e-03) ✅   
33   /connection (1.96e-03) ✅       /object (1.75e-03) ✅   
34          /job (1.90e-03) ✅   /connection (1.71e-03) ✅   
35          task (1.84e-03) ✅      /logging (1.66e-03) ✅   
36         break (1.78e-03) ✅              we (1.63e-03)   
37      solution (1.77e-03) ✅          task (1.62e-03) ✅   
38        decode (1.49e-03) ✅           /pr (1.58e-03) ✅   
39       /engine (1.34e-03) ✅      solution (1.50e-03) ✅   
40       /object (1.27e-03) ✅          /con (1.26e-03) ✅   
41           /pr (1.27e-03) ✅          step (1.18e-03) ✅   
42        begins (1.19e-03) ✅       analyze (1.14e-03) ✅   
43  /environment (1.17e-03) ✅  /application (1.13e-03) ✅   
44       /shared (1.16e-03) ✅        /legal (1.11e-03) ✅   
45      /effects (1.15e-03) ✅      /effects (1.07e-03) ✅   
46          step (1.14e-03) ✅         appears (9.97e-04)   
47               : (1.11e-03)      involves (9.86e-04) ✅   
48              we (1.09e-03)       /shared (9.39e-04) ✅   
49          .Round (1.08e-03)          /man (9.10e-04) ✅   
50     /activity (1.02e-03) ✅        tackle (8.97e-04) ✅   
51        /legal (1.01e-03) ✅        prompt (8.92e-04) ✅   
52       /entity (9.77e-04) ✅         break (7.99e-04) ✅   
53      involves (9.42e-04) ✅       example (7.75e-04) ✅   
54        answer (8.98e-04) ✅          math (7.16e-04) ✅   
55            this (8.27e-04)         steps (6.82e-04) ✅   
56       example (7.34e-04) ✅      problems (6.49e-04) ✅   
57      requires (7.29e-04) ✅          /reg (6.44e-04) ✅   
58          math (6.95e-04) ✅      /testing (5.97e-04) ✅   
59        prompt (6.92e-04) ✅  /environment (5.96e-04) ✅   
60        puzzle (6.75e-04) ✅      WHATSOEVER (5.68e-04)   
61      /testing (6.52e-04) ✅     /activity (5.66e-04) ✅   
62          /reg (5.98e-04) ✅     /customer (5.15e-04) ✅   
63  /application (5.63e-04) ✅       /dialog (5.04e-04) ✅   
64      /respond (5.58e-04) ✅       /entity (5.04e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [17]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                              \
                    pos_-3                 pos_-1                pos_0   
0            Noon (0.0116)   -scalable (7.93e-03)        iban (0.0211)   
1       UNCTION (9.03e-03)      anmeld (7.45e-03)      cate (8.85e-03)   
2           aca (4.82e-03)     criptor (6.16e-03)     algun (7.32e-03)   
3           gib (4.55e-03)       zenia (5.10e-03)      maal (6.47e-03)   
4            ền (4.27e-03)       ***\n (4.24e-03)     asign (4.73e-03)   
5           cox (4.27e-03)        axis (3.74e-03)      inal (4.18e-03)   
6           Cul (3.75e-03)        PURE (3.51e-03)     iable (4.18e-03)   
7         inite (3.75e-03)        avra (3.51e-03)   InRange (3.68e-03)   
8           cco (3.54e-03)        Plus (3.30e-03)     ivery (3.68e-03)   
9         Lilly (3.54e-03)       cakes (3.10e-03)  enerated (3.46e-03)   
10            � (3.33e-03)      SignUp (2.73e-03)     ating (3.46e-03)   
11        oenix (3.11e-03)    /Library (2.73e-03)      Pert (3.46e-03)   
12          Hao (2.75e-03)       anoia (2.56e-03)     onces (3.05e-03)   
13        crete (2.58e-03)         ={` (2.41e-03)       nde (2.87e-03)   
14  .repository (2.43e-03)  Executable (2.41e-03)     itere (2.87e-03)   
15          rix (2.29e-03)        VICE (2.27e-03)       LAG (2.69e-03)   
16     cohesive (2.29e-03)        ERVE (2.00e-03)    averse (2.53e-03)   
17         iola (2.01e-03)  /Documents (2.00e-03)       ixa (2.53e-03)   
18         indo (1.89e-03)        iban (1.88e-03)     ishly (2.23e-03)   
19         Cafe (1.89e-03)         сыл (1.76e-03)    ileged (2.09e-03)   

                                                                            \
                     pos_1                   pos_2                   pos_3   
0            iban (0.0172)          algun (0.0123)            --; (0.0166)   
1           --; (9.83e-03)            --; (0.0123)           Pert (0.0129)   
2       InRange (8.12e-03)           enci (0.0115)         enci (9.46e-03)   
3          cate (6.32e-03)         inal (8.42e-03)      eniable (9.46e-03)   
4         itere (5.95e-03)         Pert (6.56e-03)   Destructor (7.87e-03)   
5           LAG (5.25e-03)      InRange (6.56e-03)         inal (6.93e-03)   
6         algun (5.25e-03)         iban (6.16e-03)         ises (5.74e-03)   
7            иг (4.91e-03)        onces (5.77e-03)        algun (5.74e-03)   
8      enerated (4.91e-03)         ises (5.77e-03)        onces (5.40e-03)   
9          dana (4.64e-03)         wish (4.52e-03)      InRange (5.07e-03)   
10         Pert (4.33e-03)         elow (4.24e-03)         iban (4.76e-03)   
11       averse (4.33e-03)        regor (3.30e-03)           tu (4.49e-03)   
12         inal (3.39e-03)   Destructor (3.30e-03)         wish (3.94e-03)   
13         elow (3.39e-03)        birth (3.10e-03)         elow (3.27e-03)   
14          ixa (3.19e-03)         utos (2.91e-03)        itere (3.07e-03)   
15         bite (2.99e-03)        antor (2.91e-03)         iado (3.07e-03)   
16          GRE (2.99e-03)      eniable (2.91e-03)         lien (2.72e-03)   
17   Destructor (2.99e-03)          LAG (2.73e-03)         dana (2.55e-03)   
18         iado (2.81e-03)          рам (2.56e-03)        antor (2.55e-03)   
19     Bookmark (2.81e-03)        arine (2.56e-03)        arine (2.40e-03)   

                                                                            \
                    pos_10                  pos_50                 pos_100   
0            ises (0.0337)        eniable (0.0334)        eniable (0.0238)   
1             --; (0.0170)           ises (0.0295)           ises (0.0238)   
2         eniable (0.0159)          Await (0.0109)        Await (9.89e-03)   
3    Destructor (8.54e-03)          birth (0.0102)         PLIC (8.73e-03)   
4         Await (8.54e-03)         PLIC (9.58e-03)        birth (7.72e-03)   
5         onces (5.86e-03)       orious (5.13e-03)         anus (6.81e-03)   
6        orious (5.86e-03)        onces (4.52e-03

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [19]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                                 \
                    pos_-3                 pos_-1                   pos_0   
0              to (0.0423)         act (0.0806) ✅             Ab (0.0154)   
1          conf (0.0225) ✅          si (6.91e-03)             Se (0.0117)   
2       program (0.0215) ✅         UNT (5.28e-03)      heroine (0.0105) ✅   
3             min (0.0194)        segu (5.18e-03)         soft (8.67e-03)   
4               : (0.0186)           h (5.08e-03)     heroes (8.62e-03) ✅   
5              je (0.0117)           # (4.87e-03)         even (7.15e-03)   
6             : (9.56e-03)       AGR (4.39e-03) ✅    Reviews (6.95e-03) ✅   
7        system (8.93e-03)    REVIEW (3.89e-03) ✅            k (6.13e-03)   
8     session (5.98e-03) ✅         LAG (3.59e-03)         Port (5.81e-03)   
9          will (5.78e-03)        RE (3.50e-03) ✅            v (5.69e-03)   
10          Geo (5.46e-03)       cit (3.49e-03) ✅          set (5.47e-03)   
11           Re (4.93e-03)          ig (2.95e-03)     review (5.24e-03) ✅   
12           bo (4.91e-03)       cat (2.69e-03) ✅       CONFIG (5.23e-03)   
13       Cong (4.71e-03) ✅       agt (2.51e-03) ✅       Silver (5.17e-03)   
14          Pan (4.64e-03)   RELEASE (2.36e-03) ✅    reviews (5.03e-03) ✅   
15   confession (4.63e-03)          re (2.36e-03)           se (4.33e-03)   
16            , (4.50e-03)      DATE (2.23e-03) ✅     skeletal (4.11e-03)   
17           Fe (4.49e-03)        po (2.20e-03) ✅       hero (4.04e-03) ✅   
18        congr (4.24e-03)       .open (2.10e-03)     Review (3.97e-03) ✅   
19        Con (4.06e-03) ✅          fo (1.96e-03)   reviewed (3.74e-03) ✅   

                                                            \
                      pos_1                          pos_2   
0     characters (0.1255) ✅                    -> (0.2216)   
1               -> (0.0971)                '\n\n' (0.0304)   
2               to (0.0367)               :\n\n (7.57e-03)   
3                a (0.0335)        characters (7.46e-03) ✅   
4                : (0.0281)         fictional (7.13e-03) ✅   
5      character (0.0214) ✅                 ' ' (5.92e-03)   
6              the (0.0196)                   : (5.41e-03)   
7              ' ' (0.0188)                   , (4.63e-03)   
8                : (0.0125)                from (4.32e-03)   
9                , (0.0121)                 you (4.20e-03)   
10             you (0.0116)        repetition (4.16e-03) ✅   
11    dialogue (9.36e-03) ✅              text (3.11e-03) ✅   
12   fictional (7.98e-03) ✅    representation (3.09e-03) ✅   
13         two (7.92e-03) ✅                  in (2.96e-03)   
14        text (7.82e-03) ✅   representations (2.81e-03) ✅   
15           all (5.99e-03)                '\n' (2.81e-03)   
16          some (5.61e-03)          dialogue (2.71e-03) ✅   
17          your (5.03e-03)                   : (2.66e-03)   
18          date (4.47e-03)            dialog (2.64e-03) ✅   
19      dialog (4.25e-03) ✅             results (2.39e-03)   

                                                 layer_14  \
                        pos_3                      pos_-3   
0                 -> (0.1219)           prints (0.0175) ✅   
1             '\n\n' (0.0180)             city (0.0162) ✅   
2       repetition (0.0160) ✅            print (0.0144) ✅   
3              you (7.09e-03)              super (0.0117)   
4        silence (5.08e-03) ✅           memory (0.0117) ✅   
5      sequences (4.95e-03) ✅                  B (0.0113)   
6     characters (4.45e-03) ✅           city (5.89e-03) ✅   
7            :\n\n (4.43e-03)              som (5.29e-03)   
8              ' ' (3.43e-03)           rows (4.97e-03) ✅   
9               in (3.25e-03)         cities (4.35e-03) ✅   
10      dialogue (2.94e-03) ✅   dictionaries (4.09e-03) ✅   
11               f (2.79e-03)           bots (4.06e-03) ✅   
12    repetitive (2.64e-03) ✅        results (3.65e-03) ✅   
13         numbers (2.60e-03)       memories